# Stage1 SOXL 目标权重与相对 SOXX 收益排序

复用冻结 A 完整策略 09:46 目标和原训练价格面 H1/H5 SOXL−SOXX 回报；940 条成熟 H5 行为主。仅只读信息诊断，不训练、不修改持仓、不跑账户。

In [1]:
import os,json,hashlib,pickle,tarfile,tempfile,subprocess,shlex
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from scipy.stats import spearmanr
from src.data.nas_market_data import archive_nas_recorder,_settings
root=Path.cwd();card=root/'research/stage1_soxl_relative_target_information_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists()
a=plan['stage1_source'];b=plan['label_source'];assert sha(root/a['summary'])==a['summary_sha256']
assert sha(root/'research/stage1_domain_heads_20261018/qlib_recorder'/a['experiment_id']/a['recorder_id']/'artifacts'/a['artifact'])==a['artifact_sha256']
for name,key in [('inputs/daily_frame','daily_frame_sha256'),('inputs/timing','timing_sha256')]:assert sha(root/b['card']/'qlib_recorder'/b['experiment_id']/b['recorder_id']/'artifacts'/name)==b[key]
state={'status':'started','source_Recorder_object_loads':0,'reused_label_objects':0,'bootstrap_attempted':False,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/plan['provider']),region='us',kernels=1,expression_cache=None,dataset_cache=None)
s=json.loads((root/a['summary']).read_text())
with R.uri_context(s['recorder_uri']):paths=R.get_recorder(recorder_id=a['recorder_id'],experiment_id=a['experiment_id']).load_object(a['artifact'])
state['source_Recorder_object_loads']+=1
with R.uri_context(str(root/b['card']/'qlib_recorder')):
 rec=R.get_recorder(recorder_id=b['recorder_id'],experiment_id=b['experiment_id']);frame=rec.load_object('inputs/daily_frame');timing=rec.load_object('inputs/timing')
state['source_Recorder_object_loads']+=2;state['reused_label_objects']=2;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
targets=paths['targets'].copy();assert len(targets)==944 and targets.signal_date.is_unique and frame.index.equals(timing.index)
calendar=pd.DatetimeIndex(D.calendar(start_time=plan['cohort']['signal_dates'][0],end_time=plan['cohort']['signal_dates'][1],freq='day'));assert len(calendar)==944 and set(pd.to_datetime(targets.signal_date))==set(calendar)
soxl=frame.soxl_forward_open_return.astype(float);soxx=frame.soxx_forward_open_return.astype(float)
valid1=np.isfinite(soxl)&np.isfinite(soxx)&soxl.gt(-1)&soxx.gt(-1)
relative_h1=pd.Series(np.where(valid1,np.log1p(soxl)-np.log1p(soxx),np.nan),index=frame.index,name='SOXL_minus_SOXX_H1_log')
parts=[relative_h1.shift(-i) for i in range(5)];valid5=np.logical_and.reduce([np.isfinite(x.to_numpy(float)) for x in parts]);relative_h5=pd.Series(pd.concat(parts,axis=1).sum(axis=1).where(valid5),index=frame.index,name='SOXL_minus_SOXX_H5_log')
cutoff=pd.Timestamp('2026-10-10',tz='America/New_York').tz_convert('UTC');available1=pd.to_datetime(timing.return_available_at,utc=True);available5=pd.to_datetime(timing.return_available_at.shift(-4),utc=True)
dates=pd.DatetimeIndex(pd.to_datetime(targets.signal_date));assert dates.equals(calendar);score=pd.Series(targets.soxl_weight.to_numpy(float),index=dates,name='Stage1_SOXL_target_weight')
matched=pd.DataFrame({'signal_date':dates,'trade_date':pd.to_datetime(targets.trade_date),'target_weight':score.to_numpy(float),'relative_H1':relative_h1.reindex(dates).to_numpy(float),'relative_H5':relative_h5.reindex(dates).to_numpy(float),'H1_available_at':available1.reindex(dates).to_numpy(),'H5_available_at':available5.reindex(dates).to_numpy()})
matched['H1_mature']=matched.relative_H1.notna()&(pd.to_datetime(matched.H1_available_at,utc=True)<cutoff)
matched['H5_mature']=matched.relative_H5.notna()&(pd.to_datetime(matched.H5_available_at,utc=True)<cutoff)
assert len(matched)==944 and matched.signal_date.is_unique and np.isfinite(matched.target_weight).all()
assert int(matched.H1_mature.sum())==944 and int(matched.H5_mature.sum())==940
static=matched.set_index('signal_date')[['target_weight','relative_H1','relative_H5','H1_mature','H5_mature']]
static.index=pd.MultiIndex.from_arrays([calendar,['SOXL_SOXX_CASH']*len(calendar)],names=['datetime','instrument'])
static=pd.concat({'feature':static[['target_weight']],'label':static[['relative_H1','relative_H5']],'meta':static[['H1_mature','H5_mature']]},axis=1)
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'held':tuple(plan['cohort']['signal_dates'])})
assert len(dataset.prepare('held',col_set=['feature','label','meta']))==944
h1=matched[matched.H1_mature];h5=matched[matched.H5_mature];rho5=float(spearmanr(h5.target_weight,h5.relative_H5).statistic);rho1=float(spearmanr(h1.target_weight,h1.relative_H1).statistic)
score5=h5.target_weight.to_numpy(float);label5=h5.relative_H5.to_numpy(float);rng=np.random.default_rng(20261023);n=len(h5);reps=[];state['bootstrap_attempted']=True;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
for _ in range(2000):
 idx=np.empty(n,dtype=int);idx[0]=rng.integers(n)
 for j in range(1,n):idx[j]=rng.integers(n) if rng.random()<1/63 else (idx[j-1]+1)%n
 reps.append(float(spearmanr(score5[idx],label5[idx]).statistic))
reps=np.array(reps);assert np.isfinite(reps).all();ci=np.quantile(reps,[.025,.975])
annual=[];quarters=[];named=[]
for period,g in h5.groupby(h5.signal_date.dt.year):annual.append({'year':int(period),'mature_rows':len(g),'mean_target_weight':float(g.target_weight.mean()),'mean_relative_H5_log':float(g.relative_H5.mean()),'target_vs_relative_H5_spearman':float(spearmanr(g.target_weight,g.relative_H5).statistic) if len(g)>10 else None})
for period,g in h5.groupby(h5.signal_date.dt.to_period('Q').astype(str)):quarters.append({'quarter':period,'mature_rows':len(g),'mean_target_weight':float(g.target_weight.mean()),'mean_relative_H5_log':float(g.relative_H5.mean()),'target_vs_relative_H5_spearman':float(spearmanr(g.target_weight,g.relative_H5).statistic) if len(g)>10 else None})
for name,window in plan['named_regimes'].items():
 g=h5[(h5.signal_date>=pd.Timestamp(window[0]))&(h5.signal_date<=pd.Timestamp(window[1]))]
 named.append({'regime':name,'signal_start':window[0],'signal_end':window[1],'mature_rows':len(g),'mean_target_weight':float(g.target_weight.mean()),'mean_relative_H5_log':float(g.relative_H5.mean()),'target_vs_relative_H5_spearman':float(spearmanr(g.target_weight,g.relative_H5).statistic) if len(g)>10 else None})
report={'status':'frozen_Stage1_SOXL_target_relative_return_rank_diagnostic_only','plan_sha256':sha(card/'plan.json'),'matched_account_dates':944,'mature_H1_rows':944,'mature_H5_rows':940,'target_weight_min':float(matched.target_weight.min()),'target_weight_max':float(matched.target_weight.max()),'H1_spearman':rho1,'H5_primary_spearman':rho5,'H5_descriptive_95pct_block_interval':[float(ci[0]),float(ci[1])],'annual':annual,'quarters':quarters,'named_regimes':named,'bootstrap_replicates':2000,'model_fits':0,'native_accounts':0,'reused_history':True,'target_is_pure_alpha_probability':False,'full_account_counterfactual':False}
(card/'source_audit.json').write_text(json.dumps(report,indent=2)+'\n')
dataset.config(dump_all=True,recursive=True);assert len(pickle.loads(pickle.dumps(dataset)).prepare('held',col_set=['feature','label','meta']))==944
tracking=card/'qlib_recorder'
with R.start(experiment_name='Stage1_SOXL_target_relative_return_rank_diagnostic',uri=str(tracking)):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,source_hashes={'Stage1_path':a['artifact_sha256'],'daily_frame':b['daily_frame_sha256'],'timing':b['timing_sha256']})
 rec.save_objects(artifact_path='analysis',matched_daily=matched,annual=annual,quarters=quarters,named_regimes=named,bootstrap_replicates=reps,source_audit=report)
 rec.save_objects(artifact_path='metadata',dataset=dataset)
 rec.log_metrics(mature_H5_rows=940,H5_rank_spearman=rho5,H5_rank_lower=float(ci[0]),model_fits=0,native_accounts=0)
 recorder_id=rec.id;experiment_id=str(rec.experiment_id)
with R.uri_context(str(tracking)):
 checked=R.get_recorder(recorder_id=recorder_id,experiment_id=experiment_id);assert len(checked.load_object('analysis/matched_daily'))==944 and len(checked.load_object('metadata/dataset').prepare('held',col_set=['feature','label','meta']))==944
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':'Stage1_SOXL_target_relative_return_rank_diagnostic','ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=recorder_id,experiment_id=experiment_id)
with tempfile.TemporaryDirectory(prefix='soxl-relative-rank-restore-') as temp:
 archive=Path(temp)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(evidence['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==evidence['sha256']
 with tarfile.open(archive,'r:gz') as source:names=[m.name for m in source.getmembers() if m.isfile()]
 assert any(name.endswith('/artifacts/metadata/dataset') for name in names)
summary={'experiment':card.name,'decision':'diagnostic_only_frozen_Stage1_SOXL_target_relative_information_no_account','objective_achieved':False,'plan':'plan.json','source_audit':'source_audit.json','model_fits':0,'native_accounts':0,'recorder_id':recorder_id,'experiment_id':experiment_id,'recorder_uri':str(tracking),'nas_evidence':evidence,'nas_readback':{'sha256':evidence['sha256'],'verified':True},'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/*','metadata/*']}}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
state['status']='completed';state['recorder_id']=recorder_id;state['nas_archive_sha256']=evidence['sha256'];(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
print({'H1_spearman':rho1,'H5_spearman':rho5,'H5_interval':[float(ci[0]),float(ci[1])],'named_regimes':named,'recorder_id':recorder_id,'nas_sha256':evidence['sha256']})


{'H1_spearman': -0.013569445118868631, 'H5_spearman': -0.04492366931125242, 'H5_interval': [-0.21356661103528224, 0.08190292598560087], 'named_regimes': [{'regime': 'cash_helped_2024Q3', 'signal_start': '2024-07-01', 'signal_end': '2024-09-30', 'mature_rows': 64, 'mean_target_weight': 0.08604734030711486, 'mean_relative_H5_log': -0.03313435584204007, 'target_vs_relative_H5_spearman': 0.0660548352580135}, {'regime': 'worst_126_cash_drag', 'signal_start': '2025-04-07', 'signal_end': '2025-10-07', 'mature_rows': 127, 'mean_target_weight': 0.17511098071846873, 'mean_relative_H5_log': 0.03351922584693036, 'target_vs_relative_H5_spearman': -0.1738675845338634}, {'regime': 'cash_and_SOXL_selection_weak_2026Q3', 'signal_start': '2026-07-01', 'signal_end': '2026-09-30', 'mature_rows': 64, 'mean_target_weight': 0.025030079229178628, 'mean_relative_H5_log': -0.013723785775581634, 'target_vs_relative_H5_spearman': -0.1075745199963193}], 'recorder_id': '7e833a37c0e84e25a4c4c46c624f9141', 'nas_sha25

## 中文结论

冻结 Stage1 09:46 风控后 SOXL 目标权重对原训练价格面的未来 SOXL−SOXX H1/H5 相对收益排序为 -0.0136/-0.0449，H5 描述性区块区间 [-0.2136,0.0819] 跨 0。按原账户 trade_date 半开边界修复后的最差 126 日，权重与 H5 相对收益相关 -0.1817；原信号日 127 行不能直接称为账户 126 日。正的 SOXL 相对会计贡献不证明预测排序，来源价格面也不替代真实 Tiered 账户。DatasetH/Recorder `7e833a37c0e84e25a4c4c46c624f9141` 与 NAS SHA-256 `22bf503f4dcc780e97341548952650ca13da6b7e66cb3b895c982c99d0fd6e18` 回读通过。0 拟合、0 新账户，决定仅诊断、不晋升。